# Enhanced Cleaning — CNBC
Remove trailing boilerplate: `Selengkapnya saksikan ...`

In [ ]:
import re
import pandas as pd

DATA_PATH   = "/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled"
OUTPUT_PATH = "/kaggle/working"

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/cnbc_labeled.csv")
print(f"Loaded: {df.shape}")
df.head(3)

## EDA — Before Cleaning

In [ ]:
print("=== Label Distribution ===")
print(df["label"].value_counts())
print(f"\nAvg content length : {df['content'].str.len().mean():.0f} chars")
hit = df["content"].str.contains(r"Selengkapnya saksikan", case=False, na=False).sum()
print(f"Rows containing 'Selengkapnya saksikan': {hit} / {len(df)}")

## Cleaning — Truncate at `Selengkapnya saksikan`

In [ ]:
# Potong semua teks mulai dari trigger phrase (inklusif)
CNBC_CUT_PATTERNS = [
    r"Selengkapnya saksikan[^\n]*",
]

def truncate_cnbc(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    for pattern in CNBC_CUT_PATTERNS:
        m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            text = text[:m.start()].rstrip()
    return re.sub(r"\s+", " ", text).strip()

df["content_clean"] = df["content"].apply(truncate_cnbc)
affected = (df["content_clean"] != df["content"].fillna("")).sum()
print(f"Done. Rows affected: {affected} / {len(df)}")

## Before / After Comparison

In [ ]:
mask = df["content"].str.contains(r"Selengkapnya saksikan", case=False, na=False)
for _, row in df[mask].head(2).iterrows():
    print("=== BEFORE (last 300 chars) ===")
    print(str(row["content"])[-300:])
    print("\n=== AFTER (last 300 chars) ===")
    print(str(row["content_clean"])[-300:])
    print("-" * 70)

print(f"\nAvg length BEFORE : {df['content'].str.len().mean():.0f}")
print(f"Avg length AFTER  : {df['content_clean'].str.len().mean():.0f}")

## Rebuild `text` = title + content_clean

In [ ]:
df["text"] = (
    df["title"].astype(str).str.strip() + ". " +
    df["content_clean"].astype(str).str.strip()
).str.strip()

df_out = df[["date", "title", "content_clean", "article_id", "text", "label"]].copy()
df_out = df_out.rename(columns={"content_clean": "content"})

print(f"Output shape : {df_out.shape}")
print(df_out["label"].value_counts())
df_out.head(3)

In [ ]:
df_out.to_csv(f"{OUTPUT_PATH}/cnbc_labeled.csv", index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}/cnbc_labeled.csv  ({len(df_out)} rows)")